# Travel Agent - Trip Planning System

This notebook contains a comprehensive travel planning system using MCP (Model Context Protocol) servers with AI agents for trip planning, flight finding, and activity planning.

In [1]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from pydantic import BaseModel
from typing import List, Dict
from datetime import datetime, timedelta

load_dotenv(override=True)

True

In [2]:
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}
]
mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in mcp_server_params]

## Data Models

In [4]:
class Destination(BaseModel):
    city: str
    days: int
    

class TripDividerOutput(BaseModel):
    Destinations: List[Destination]


class FlightFinderOutput(BaseModel):
    Airline: str
    Departure: str
    Arrival: str
    Departure_Time: str
    Arrival_Time: str
    Price: str
    Link: str


class FlightList(BaseModel):
    flights: List[FlightFinderOutput]


class ActivityPlannerOutput(BaseModel):
    Food: str
    Activities: str

## Planning Agent

In [5]:
async def get_trip_divider(mcp_servers) -> Agent:
    instructions = f"""You are an trip planner. You will determine the number of days to stay in each city. 
    The number of days in each city should depend on how popular the city. For example, you will spend more days in London than in manchester
    as london offers more stuff to do.
    You will also determine the order in which cities are visited. The order of the cities in the list represents the order of the cities visted.
    Output a list where each element is the city and the number of days to spend in that city the start and end dates of the trip will be given).
    Do not put starting city in the output.
    Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
    """
    trip_divider = Agent(
        name="trip_divider",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
       output_type=TripDividerOutput,
    )
    return trip_divider

## Flight Agent

In [6]:
async def get_flight_finder(mcp_servers) -> Agent:
    instructions = f"""You are a flight finder. You are able to search the web for flight details.
Based on the request, you carry out necessary research and respond with your findings.
The flight should be on the date provided and the start and end destination should be the ones provided.
Return a link to the booking website where the result was found
Give 1 flight option or max 2 flight options
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
If there isn't a specific request, then just respond with "NA
"""
    flight_finder = Agent(
        name="flight_finder",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
        output_type=FlightList,
    )
    return flight_finder

In [7]:
async def get_flights_for_trip(flight_schedule, mcp_servers):
    """Get flights for all legs of the trip"""
    all_results: List[FlightList] = []
    
    # Get agent
    flight_finder = await get_flight_finder(mcp_servers)
    
    # Loop through each leg
    for origin, destination, date in flight_schedule:
        question = f"flights from {origin} to {destination} on {date}"
        with trace("flight_finder"):
            result = await Runner.run(flight_finder, question, max_turns=30)
            all_results.append(result.final_output)
    
    return all_results

## Activity Agent

In [8]:
async def get_activity_planner(mcp_servers) -> Agent:
    instructions = f"""You are an activity planner. You are able to search the web for activities and food.
Based on the request, you carry out necessary research and respond with your findings.
The activities and food should be in the destination provided.
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
"""
    activity_planner = Agent(
        name="activity_planner",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
        output_type=ActivityPlannerOutput,
    )
    return activity_planner

## Complete Trip Planning Function

In [9]:
async def plan_complete_trip(start_date, end_date, start_destination, destinations, additional_info=""):
    """
    Complete trip planning function that combines all agents
    """
    
    # Connect to MCP servers
    for server in mcp_servers:
        await server.connect()
    
    # 1. Plan trip divisions
    question = f"Determine the number of days to stay in each city and the order to visit them. Start from {start_destination} to {destinations[0]} on {start_date} and return on {end_date} with the following additional information: {additional_info}."
    
    trip_divider = await get_trip_divider(mcp_servers)
    with trace("trip_divider"):
        trip_result = await Runner.run(trip_divider, question, max_turns=30)
    
    # 2. Create flight schedule
    travel_schedule = [[dest.city, dest.days] for dest in trip_result.final_output.Destinations]
    
    date = datetime.strptime(start_date, "%Y-%m-%d")
    flight_schedule = []
    current = start_destination
    
    for city, stay_days in travel_schedule:
        flight_schedule.append([current, city, date.strftime("%Y-%m-%d")])
        current = city
        date += timedelta(days=stay_days)
    
    flight_schedule.append([current, start_destination, date.strftime("%Y-%m-%d")])
    
    # 3. Get flights
    all_flights = await get_flights_for_trip(flight_schedule, mcp_servers)
    
    # 4. Get activities for each destination
    activity_planner = await get_activity_planner(mcp_servers)
    activities_results = []
    
    for destination in trip_result.final_output.Destinations:
        question = f"Plan things to do in {destination.city} for {destination.days} days"
        with trace("activity_planner"):
            activity_result = await Runner.run(activity_planner, question, max_turns=30)
            activities_results.append({
                'city': destination.city,
                'days': destination.days,
                'activities': activity_result.final_output
            })
    
    return {
        'trip_plan': trip_result.final_output,
        'flight_schedule': flight_schedule,
        'flights': all_flights,
        'activities': activities_results
    }

In [10]:
def display_trip_plan(trip_data, start_date, end_date, start_destination):
    print("\n" + "="*60)
    print("           TRIP ITINERARY")
    print("="*60)

    print(f"\n{start_date} to {end_date}")
    print(f"Starting from: {start_destination}\n")

    print("DESTINATIONS:")
    for dest in trip_data['trip_plan'].Destinations:
        print(f"   • {dest.city}: {dest.days} days")

    print("\nFLIGHTS:")
    for i, (origin, dest, date) in enumerate(trip_data['flight_schedule'], 1):
        print(f"\n   Flight {i}: {origin} → {dest} ({date})")
        if i-1 < len(trip_data['flights']) and trip_data['flights'][i-1].flights:
            for flight in trip_data['flights'][i-1].flights:
                print(f"      • {flight.Airline}: {flight.Price}")
                print(f"        Departure: {flight.Departure_Time}")
                print(f"        Arrival: {flight.Arrival_Time}")
                print(f"        Link: {flight.Link}")

    print("\n ACTIVITIES:")
    for activity in trip_data['activities']:
        print(f"\n   {activity['city'].upper()} ({activity['days']} days)")
        print(f"   Food: {activity['activities'].Food}")
        print(f"   Activities: {activity['activities'].Activities}")
    
    print("\n" + "="*60)

In [ ]:
# ============================================
# EDIT YOUR TRIP DETAILS HERE
# ============================================

start_date = "2025-10-01"           # Format: YYYY-MM-DD
end_date = "2025-10-10"             # Format: YYYY-MM-DD
start_destination = "Toronto"       # Your home city/starting point
destinations = ["Nova Scotia"]      # List of cities you want to visit (can add multiple: ["Paris", "London", "Rome"])
additional_info = ""                # Any preferences (e.g., "budget travel", "luxury hotels", "vegetarian food")

In [ ]:
# Run the complete trip planning
trip_data = await plan_complete_trip(
    start_date=start_date,
    end_date=end_date,
    start_destination=start_destination,
    destinations=destinations,
    additional_info=additional_info
)

# Display the results
display_trip_plan(trip_data, start_date, end_date, start_destination)